# Guide: CIP Premium and Regime Sensitivity

Computes raw/purified CIP basis, then applies a deterministic rolling term-premium model and regime comparison.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().resolve().parent))

from src.analytics.cip_premium import (
    compute_raw_cip_deviation,
    compute_purified_cip_deviation,
    TermPremiumModel,
    regime_sensitivity,
)


In [ ]:
np.random.seed(77)
dates = pd.date_range('2025-01-01', periods=80, freq='B')
tenors = [1.0, 2.0, 5.0]

spot = pd.Series(360 + np.cumsum(np.random.normal(0, 0.2, len(dates))), index=dates)
dom_ois = pd.DataFrame({t: 0.06 + 0.001*t + np.random.normal(0, 0.0002, len(dates)) for t in tenors}, index=dates)
for_ois = pd.DataFrame({t: 0.04 + 0.0007*t + np.random.normal(0, 0.00015, len(dates)) for t in tenors}, index=dates)

forwards = pd.DataFrame(index=dates, columns=tenors, dtype=float)
for t in tenors:
    forwards[t] = spot * (1 + dom_ois[t]*t) / (1 + for_ois[t]*t)

# Inject macro stress period with larger forward dislocation.
stress_slice = dates[-20:]
forwards.loc[stress_slice, 2.0] *= 1.0015
forwards.loc[stress_slice, 5.0] *= 1.0025

raw = compute_raw_cip_deviation(spot, forwards, dom_ois, for_ois)
raw_basis = raw['raw_basis_bp']

# Sovereign/supranational spreads chosen to create larger credit differential during stress.
dom_sov = dom_ois + 0.018
dom_sup = dom_ois + 0.010
for_sov = for_ois + 0.008
for_sup = for_ois + 0.006

dom_sov.loc[stress_slice] += 0.010
dom_sup.loc[stress_slice] += 0.003

purified = compute_purified_cip_deviation(raw_basis, dom_sov, for_sov, dom_sup, for_sup)
purified.tail()


In [ ]:
# Rolling regime model on 5Y purified basis.
y = purified['purified_basis_bp'][5.0] / 1e4
X = pd.DataFrame({
    'risk_off': np.linspace(0.0,1.0,len(dates)) + np.random.normal(0,0.03,len(dates)),
    'usd_funding': np.linspace(0.2,0.7,len(dates)) + np.random.normal(0,0.02,len(dates)),
    'fx_vol': np.concatenate([np.full(len(dates)-20,0.18), np.full(20,0.30)]),
}, index=dates)

fit = TermPremiumModel(intercept=True).rolling_window_estimation(X, y, window=40, min_obs=24)
coefs = fit['coefficients']
regime_flag = pd.Series((X['fx_vol'] > 0.24).astype(int), index=dates)
regime_cmp = regime_sensitivity(coefs, regime_flag)
regime_cmp


In [ ]:
class CIPExplainer:
    def __init__(self, purified: pd.DataFrame, regime_cmp: pd.DataFrame):
        self.purified = purified
        self.regime_cmp = regime_cmp

    def render_full_markdown(self) -> str:
        rb = self.purified['raw_basis_bp'][5.0].iloc[-20:].mean()
        pb = self.purified['purified_basis_bp'][5.0].iloc[-20:].mean()
        delta = rb - pb
        fx_vol_beta_shift = float(self.regime_cmp.loc['fx_vol','delta_high_minus_low']) if 'fx_vol' in self.regime_cmp.index else float('nan')
        return f"""
### Interpretation (P&L / DV01 / Convexity / Basis / Hedge Action)
- **Macro/regime change:** stressed window shows materially wider raw CIP basis at 5Y.
- **Purification effect:** credit-cleaning removes about **{delta:.1f} bp** on average from stressed raw basis.
- **P&L impact:** basis-sensitive books (FX swaps, xccy swaps) see P&L dominated by basis widening rather than pure rates.
- **DV01 & convexity:** standard rates DV01 hedges only partly protect when CIP basis shifts nonlinearly with risk-off factors.
- **Regime sensitivity:** estimated change in `fx_vol` coefficient is about **{fx_vol_beta_shift:.4f}**.
- **Hedge action:** combine rates DV01 hedges with basis-specific hedges and monitor regime flags for hedge-ratio recalibration.
""".strip()

explainer = CIPExplainer(purified, regime_cmp)
print(explainer.render_full_markdown())
